In [ ]:
import numpy as np

X_train_70 = np.load('X_train.npy')
y_train_70 = np.load('y_train.npy')

X_val_15 = np.load('X_val.npy')
y_val_15 = np.load('y_val.npy')

X_cv_85 = np.concatenate((X_train_70, X_val_15), axis=0)
y_cv_85 = np.concatenate((y_train_70, y_val_15), axis=0)

np.save('X_cv_85.npy', X_cv_85)
np.save('y_cv_85.npy', y_cv_85)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

def new_lstm_model():
    
    model = Sequential()
    model.add(Embedding(
        input_dim=5000,      
        output_dim=64,        
        input_length=70,    
        mask_zero=True 
    ))
    
    model.add(LSTM(units=32))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(
        optimizer=Adam(learning_rate=0.001),              
        loss='binary_crossentropy',    
        metrics=['accuracy']           
    )
    return model

def run_lstm_cross_validation(X, y, n_splits=5, epochs=5, batch_size=32):

    fold_each_result = []
    y_true = []
    y_pred = []
    target_name  = [0, 1]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    print(f"[ เริ่มทำ {n_splits}-Fold Cross-validation ]")
    
    # fold ทีละก้อน
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"กำลังเทรน Fold ที่ {fold_idx + 1}")
        
        X_fold_train, X_fold_val = X[train_idx], X[val_idx]
        y_fold_train, y_fold_val = y[train_idx], y[val_idx]
        
        model = new_lstm_model() # เรียกใช้โมเดลตัวใหม่
        
        history = model.fit(
            x=X_fold_train,
            y=y_fold_train,
            validation_data=(X_fold_val, y_fold_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=1
        )
        
        # ประเมินผลรอบสุดท้าย 
        loss, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        fold_each_result.append(accuracy)
        print(f"ความแม่นยำ Fold {fold_idx + 1}: {accuracy:.2f}")
        y_true.extend(y_fold_val)
        y_pred.extend((model.predict(X_fold_val) > 0.5).astype(int).flatten())
        
        
        tf.keras.backend.clear_session()
        
    # สรุปผล
    avg_accuracy = np.mean(fold_each_result)
    print(f"สรุปผล: CV Accuracy = {avg_accuracy:.2f}")
    
    return avg_accuracy, y_true, y_pred

X_cv_data = np.load('X_cv_85.npy')
y_cv_data = np.load('y_cv_85.npy')

final_score, y_true, y_pred = run_lstm_cross_validation(X_cv_data, y_cv_data)
print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

KeyboardInterrupt: 